***

Preparing Workspace

***

In [ ]:
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
import re
from datetime import date
import requests
import ast
import xlwt
from xlwt.Workbook import *
from pandas import ExcelWriter
import xlsxwriter
import time
import functools as ft
import urllib.request, json
pd.set_option('display.max_columns', None)


In [ ]:
# Define user
user = os.getlogin()
path_users = os.path.join('C:\\Users', user)

## Set file paths
if user == 'jfontes':
    # Git
    path_git = os.path.join(path_users, 'Documents', 'Projects', 'Regional-Monitoring', 'Indicator_Gen')

    # SharePoint
    path_out  = os.path.join(path_users
                             , 'Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents'
                             , 'Process Revamp'
                             , 'Task 9. Collect new data'
                             , 'EPA')

path_code    = os.path.join(path_git, 'Data', 'EPA')
path_config0 = os.path.join(path_git , 'config')
path_config  = os.path.join(path_code, 'config')

In [ ]:
## User defined functions
exec(open(os.path.join(path_config0, 'Functions.py')).read())

***

Testing

***

In [ ]:

# root_ = 'https://aqs.epa.gov/data/api/'
# email_ = 'jfontes@sacog.org'
# api_key = 'aquaram35'
# # email_ = 'test@aqs.api'
# # api_key = 'test'
# data_ = 'dailyData'
# geo_ = 'CBSA'
# param = '88101'
# year = 2021
# geo = '40900'

# url_to_import = f"{root_}{data_}/by{geo_}?email={email_}&key={api_key}&param={param}&bdate={year}0101&edate={year}1231&cbsa={geo}"
            
# print('Importing...')
# with urllib.request.urlopen(url_to_import) as url:
#     dict_aqi = json.load(url)
# df_aqi = pd.DataFrame(dict_aqi['Data'])

# print('Sleeping for 12 seconds...')
# time.sleep(12)

# df_aqi.head()


***

Importing

***

In [ ]:
start_time = time.time()


root_   = 'https://aqs.epa.gov/data/api/'
email_  = 'jfontes@sacog.org'
key_ = 'aquaram35'
data_   = 'annualData'#'dailyData'
geo_    = 'CBSA'
params  = ['88101', '44201']
years   = sequence(2021, 2022, 1)
geos    = ['40900', '49700']

list_df_geos = []

for geo in geos:
    
    print(''); print('Importing MSA:', geo); print('')
    list_df_params = []
    
    for param in params:
        
        print(''); print('Importing parameter:', param)
        list_df_years = []
        
        for year in tqdm(years):

            time.sleep(6) # Sleeping for 6 seconds between requests to not upset the EPA overlords
            
            url_to_import = f"{root_}{data_}/by{geo_}?email={email_}&key={key_}&param={param}&bdate={year}0101&edate={year}1231&cbsa={geo}"
            
            with urllib.request.urlopen(url_to_import) as url:
                dict_aqi = json.load(url)
            df_aqi = pd.DataFrame(dict_aqi['Data'])
            df_aqi['Year_Imported'] = year
            list_df_years.append(df_aqi)
            
        df_params = pd.concat(list_df_years)
        list_df_params.append(df_params)
        
    df_geo = pd.concat(list_df_params)
    list_df_geos.append(df_geo)
    print('')

df_aqi = pd.concat(list_df_geos)
df_aqi = df_aqi.reset_index(drop=True)

print("")
print("Finished!!")
print(f"Process complete.  It took --- {round((time.time() - start_time)/60, 1)} minutes ---")
print('')

df_aqi.head()

In [ ]:
df_aqi.to_excel(os.path.join(path_out, 'Health_3_MSA_EPA_annualSummary_raw.xlsx'), index=False)